In [ ]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.nn.functional as Func
from torch.distributions import Normal

import numpy as np

import random
from collections import deque

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

import os
import pickle
import shutil
import pandas as pd

import random
from collections import deque

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

### class definition ###

class _SACActor(nn.Module):
    def __init__(self):
        super(_SACActor, self).__init__()
        
        # image feature extraction
        self.map_embed = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 40x48
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 20x24
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 10x12
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 10 * 12, 512), 
            nn.ReLU(),
            nn.Linear(512, 64), 
            nn.ReLU()
        )
        
        # numerical feature extraction
        self.dashboard_embed = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # output heads after fused data
        self.mu_head = nn.Linear(128, 3)
        self.log_std_head = nn.Linear(128, 3)

    def forward(self, map_img, dash_vec):
        # extract features
        map_feat = self.map_embed(map_img)
        dash_feat = self.dashboard_embed(dash_vec)
        # fuse
        embed = torch.cat([map_feat, dash_feat], dim=1)
        
        # output
        mu = self.mu_head(embed)
        log_std = self.log_std_head(embed)
        log_std = torch.clamp(log_std, -20, 2)
        return mu, log_std

    def sample(self, map_img, dashboard_vec):
        
        # create distribution
        mu, log_std = self.forward(map_img, dashboard_vec)
        std = log_std.exp()
        dist = Normal(mu, std)
        
        # sample form distribution
        x_t = dist.rsample() 
        
        # restrict actions to -1~1 (include Change of Variables in probability)
        # $$p_Y(y) = p_X(x) \cdot \left| \frac{dx}{dy} \right|$$
        action_tanh = torch.tanh(x_t)
        log_prob = dist.log_prob(x_t)-torch.log(1-action_tanh.pow(2)+1e-6)
        
        # get actions (gas_brake from -1~1 to 0~1)
        steer = action_tanh[:, 0:1]
        gas_brake = (action_tanh[:, 1:]+1.0)/2.0
        action = torch.cat([steer, gas_brake], dim=1)
        
        # Change of Variables in probability
        correction = torch.zeros_like(log_prob)
        correction[:, 1:] = torch.log(torch.tensor(2.0))
        log_prob += correction

        return action, log_prob.sum(1, keepdim=True)

class _QNetwork(nn.Module):
    def __init__(self):
        super(_QNetwork, self).__init__()
        
        # image feature extraction
        self.map_embed = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 40x48
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 20x24
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 10x12
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 10 * 12, 512), 
            nn.ReLU(),
            nn.Linear(512, 64), 
            nn.ReLU()
        )
        
        # numerical feature extraction
        self.dashboard_embed = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # action feature extraction
        self.action_embed = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # output head after fused data
        self.q_head = nn.Sequential(
            nn.Linear(64 + 64 + 64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, map_img, dash_vec, action):
        # extract features
        map_feat = self.map_embed(map_img)
        dash_feat = self.dashboard_embed(dash_vec)
        act_feat = self.action_embed(action)
        
        # fuse
        embed = torch.cat([map_feat, dash_feat, act_feat], dim=1)
        
        # output
        q_val = self.q_head(embed)

        return q_val

class _SACDoubleCritic(nn.Module):
    def __init__(self):
        super(_SACDoubleCritic, self).__init__()
        # two independent critic
        self.Q1 = _QNetwork()
        self.Q2 = _QNetwork()

    def forward(self, map_img, dash_vec, action):
        # return two predicted Q-value to do min and update gradient individually
        q1 = self.Q1(map_img, dash_vec, action)
        q2 = self.Q2(map_img, dash_vec, action)
        return q1, q2
    
class SACAgent:
    def __init__(self, device, lr_opt=3e-4):
        self.device = device
        self.actor = _SACActor().to(device)
        self.critic = _SACDoubleCritic().to(device)
        self.target_critic = _SACDoubleCritic().to(device)
        self.target_critic.load_state_dict(self.critic.state_dict())
        
        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr_opt)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr_opt)
        
        # Adaptive entropy
        self.target_entropy = -3.0
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.alpha_opt = torch.optim.Adam([self.log_alpha], lr_opt)
        
        self.gamma = 0.99
        self.tau = 0.005

    def update_parameters(self, buffer, batch_size):
        # Sample Batch
        s_img, s_dash, a, r, ns_img, ns_dash, d, tree_idx, is_weights = buffer.sample(batch_size)        
        
        # Tensors setup
        s_img = torch.FloatTensor(s_img).permute(0, 3, 1, 2).to(self.device) / 255.0
        ns_img = torch.FloatTensor(ns_img).permute(0, 3, 1, 2).to(self.device) / 255.0

        scales = torch.FloatTensor([100.0, 230.0, 230.0, 230.0, 230.0, 1.0, 1.0, 1.0]).to(self.device)
        s_dash, ns_dash = torch.FloatTensor(s_dash).to(self.device)/scales, torch.FloatTensor(ns_dash).to(self.device)/scales
        a, r, d = torch.FloatTensor(a).to(self.device), torch.FloatTensor(r).unsqueeze(1).to(self.device), torch.FloatTensor(d).unsqueeze(1).to(self.device)
        weights = torch.FloatTensor(is_weights).unsqueeze(1).to(self.device) # �s�W�v�� Tensor

        with torch.no_grad():
            next_action, next_log_prob = self.actor.sample(ns_img, ns_dash)
            q1_t, q2_t = self.target_critic(ns_img, ns_dash, next_action)
            target_v = torch.min(q1_t, q2_t) - self.log_alpha.exp() * next_log_prob
            target_q = r + (1 - d) * self.gamma * target_v

        # update using PER weights
        curr_q1, curr_q2 = self.critic(s_img, s_dash, a)
        critic_loss1 = (weights * Func.mse_loss(curr_q1, target_q, reduction='none')).mean()
        critic_loss2 = (weights * Func.mse_loss(curr_q2, target_q, reduction='none')).mean()
        critic_loss = critic_loss1 + critic_loss2
        
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # update PER priority
        with torch.no_grad():
            td_error = (torch.abs(curr_q1 - target_q) + torch.abs(curr_q2 - target_q)) / 2.0
            td_error = td_error.cpu().numpy().flatten()
        buffer.batch_update(tree_idx, td_error)

        # Update actor
        new_a, log_prob = self.actor.sample(s_img, s_dash)
        q1_new, q2_new = self.critic(s_img, s_dash, new_a)
        actor_loss = (self.log_alpha.exp() * log_prob - torch.min(q1_new, q2_new)).mean()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        # Update Alpha
        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        for t, s in zip(self.target_critic.parameters(), self.critic.parameters()):
            t.data.copy_(t.data * (1.0 - self.tau) + s.data * self.tau)

    def save_checkpoint(self, checkpoint_path, episode, now_score, best_score):
        checkpoint = {
            'episode': episode,
            'now_score': now_score,
            'best_score': best_score,
            'actor_state_dict': self.actor.state_dict(),
            'critic_state_dict': self.critic.state_dict(),
            'target_critic_state_dict': self.target_critic.state_dict(),
            'actor_opt_state_dict': self.actor_opt.state_dict(),
            'critic_opt_state_dict': self.critic_opt.state_dict(),
            'log_alpha': self.log_alpha,
            'alpha_opt_state_dict': self.alpha_opt.state_dict()
        }
        torch.save(checkpoint, checkpoint_path)
    
    def load_checkpoint(self, checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.target_critic.load_state_dict(checkpoint['target_critic_state_dict'])
        self.actor_opt.load_state_dict(checkpoint['actor_opt_state_dict'])
        self.critic_opt.load_state_dict(checkpoint['critic_opt_state_dict'])
        self.log_alpha.data.copy_(checkpoint['log_alpha'])
        self.alpha_opt.load_state_dict(checkpoint['alpha_opt_state_dict'])
        return checkpoint['episode'], checkpoint['now_score'], checkpoint['best_score']

        
### utility function definition ###

def set_seed(seed=42):
    # Python built-in ramdom
    random.seed(seed)
    # Numpy ramdom
    np.random.seed(seed)
    # PyTorch ramdom
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    # force gpu to be deterministic (may slow down training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"seed is set to {seed}")

def preprocess(state_img, dashboard_vec, device):
    # normalize image features
    img = torch.FloatTensor(state_img.copy()).permute(2, 0, 1).unsqueeze(0) / 255.0
    
    # original upper bound of each input feature
    scales = torch.FloatTensor([100.0, 230.0, 230.0, 230.0, 230.0, 1.0, 1.0, 1.0]).to(device)
    
    # normalize numerical features
    dash = torch.FloatTensor(dashboard_vec).to(device) / scales
    
    # append batch dimension
    dash = dash.unsqueeze(0)
    
    return img.to(device), dash

def save_ReplayBuffer(buffer, path):
    with open(path, 'wb') as f:
        pickle.dump(buffer.buffer, f)

def save_PrioritizedReplayBuffer(buffer, path):
    with open(path, 'wb') as f:
        state = {
            'tree': buffer.tree.tree,
            'data': buffer.tree.data,
            'write': buffer.tree.write,
            'n_entries': buffer.tree.n_entries,
            'alpha': buffer.alpha,
            'beta': buffer.beta,
            'beta_increment': buffer.beta_increment,
            'epsilon': buffer.epsilon
        }
        pickle.dump(state, f)

def load_ReplayBuffer(buffer, path):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            buffer.buffer = pickle.load(f)

def load_PrioritizedReplayBuffer(buffer, path):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            state = pickle.load(f)
        
        buffer.tree.tree = state['tree']
        buffer.tree.data = state['data']
        buffer.tree.write = state['write']
        buffer.tree.n_entries = state['n_entries']
        buffer.alpha = state.get('alpha', buffer.alpha)
        buffer.beta = state.get('beta', buffer.beta)
        buffer.beta_increment = state.get('beta_increment', buffer.beta_increment)
        buffer.epsilon = state.get('epsilon', buffer.epsilon)
        
        return True
    else:
        return False

def evaluate_agent(agent, seed, num_trials):
    
    # initialize environment
    env_eval = gym.make("CarRacing-v3", render_mode="rgb_array", domain_randomize=False, continuous=True, max_episode_steps=5000, lap_complete_percent=0.99)
    env_eval.action_space.seed(seed)
    env_eval.observation_space.seed(seed)
    env_eval.reset(seed=seed)
    
    # reward list
    rewards = []
    
    # play {num_trials} times
    for ep in range(num_trials):
        state, info = env_eval.reset()
        for t in range(50):
            env_eval.step(np.array([0, 0, 0]))
        state = state[2:82]
        dashboard = np.zeros(8)
        tile_hit = 0
        
        for t in range(num_episode_step):
            # ��ܰʧ@
            img_t, dash_t = preprocess(state, dashboard, device)
            with torch.no_grad():
                action, _ = agent.actor.sample(img_t, dash_t)
            action = action.cpu().numpy()[0]

            # ����ʧ@
            next_state, reward, done, truncated, info = env_eval.step(action)
            next_state = next_state[2:82]
            
            # ��s���z�q (Dashboard)
            car = env_eval.unwrapped.car
            next_dashboard = np.concatenate((np.array([np.linalg.norm(car.hull.linearVelocity)] + [w.omega for w in car.wheels]), action))
            
            state, dashboard = next_state, next_dashboard
            if (reward>0):
              tile_hit += 1
        
        rewards.append(tile_hit)
    
    rewards = np.array(rewards)
    return rewards.mean(), rewards.max(), rewards.min()

if __name__ == "__main__":
    # set seed
    SEED = 67
    set_seed(SEED)
    
    # initialize environment
    env = gym.make("CarRacing-v3", render_mode="human", domain_randomize=False, continuous=True, max_episode_steps=1000, lap_complete_percent=1.0)
    env.action_space.seed(SEED)
    env.observation_space.seed(SEED)
    env.reset(seed=SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # initialize settings
    max_buffer_size = 500000
    min_buffer_size = 10000
    num_total_steps = 1000000
    num_episode_step = 5000
    batch_size = 128
    total_steps = 0
    
    # initialize buffer and agent
    agent = SACAgent(device)
    
    # about checkpoints
    version = input("version:")
    checkpoint_path = f"{version}/sac_car_best_model.pth"
    buffer_path = f"sac_car_buffer.pkl"
    now_score = 0
    best_score = float('-inf')
    mean_rewards = []
    max_rewards = []
    min_rewards = []
    
    # initialize checkpoint
    if (os.path.exists(checkpoint_path)):
        total_steps, now_score, best_score = agent.load_checkpoint(checkpoint_path)
        total_steps += 1

        print(f"Resuming from checkpoint...\n \
              step = {total_steps}\n\
              last_score = {now_score:.1f}\n\
              best_score = {best_score:.1f}\n")
    else:
        print("No saved checkpoint found, start from beginning...")
    
    # start playing
    while (True):
        state, info = env.reset()
        for t in range(50):
            env.step(np.array([0, 0, 0]))
        state = state[2:82]
        dashboard = np.zeros(8)
        episode_reward = 0
        
        for t in range(num_episode_step):
            # ��ܰʧ@
            img_t, dash_t = preprocess(state, dashboard, device)
            with torch.no_grad():
                action, _ = agent.actor.sample(img_t, dash_t)
            action = action.cpu().numpy()[0]
    
            # ����ʧ@
            next_state, reward, done, truncated, info = env.step(action)
            next_state = next_state[2:82]
            
            # ��s���z�q (Dashboard)
            car = env.unwrapped.car
            next_dashboard = np.concat((np.array([np.linalg.norm(car.hull.linearVelocity)] + [w.omega for w in car.wheels]), action))
            
            state, dashboard = next_state, next_dashboard
            episode_reward += reward
        
            total_steps += 1
    


seed is set to 67


c:\miniconda3\envs\AIracecar\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Resuming from checkpoint...
               step = 964001
              last_score = 283.0
              best_score = 283.0

